# Attention、RoPE 与缓存：逐元素检验

标准库原创机制实验，无模型、服务、GPU 或训练框架。先预测结果，再运行；断言失败也是需要保留的证据。最后生成可导入本站笔记的 JSON。

## 1. 稳定 softmax、形状与输入负例

In [ ]:
import math
def dot(a,b):
    if not a or len(a)!=len(b) or not all(math.isfinite(v) for v in a+b):
        raise ValueError('finite vectors with equal nonzero dimensions required')
    return sum(x*y for x,y in zip(a,b))
def softmax(values):
    if not values or not all(math.isfinite(v) for v in values):
        raise ValueError('finite nonempty logits required')
    largest=max(values)
    exps=[math.exp(x-largest) for x in values]
    return [x/sum(exps) for x in exps]
def rejects(fn):
    try: fn()
    except ValueError: return True
    raise AssertionError('negative case was accepted')
p=softmax([1,0,-1])
assert math.isclose(sum(p),1)
assert math.isclose(softmax([10001,10000,9999])[0],p[0])
assert rejects(lambda:softmax([]))
assert rejects(lambda:softmax([float('nan')]))
assert rejects(lambda:dot([1],[1,2]))
print('scaled weights:',p)

## 2. 完整因果 attention；改变未来不改变历史

In [ ]:
def attend(query,keys,values):
    if not keys or len(keys)!=len(values) or any(len(v)!=len(values[0]) for v in values):
        raise ValueError('matching K/V rows required')
    probs=softmax([dot(query,k)/math.sqrt(len(query)) for k in keys])
    return [sum(p*v[j] for p,v in zip(probs,values)) for j in range(len(values[0]))]
def full_causal(x):
    # Q=K=V=X: deliberately fixed identity projections, one layer only.
    return [attend(q,x[:i+1],x[:i+1]) for i,q in enumerate(x)]
x=[[1.0,0.0],[0.0,1.0],[1.0,1.0]]
full=full_causal(x)
future_changed=full_causal(x[:2]+[[100.0,-100.0]])
assert full[:2]==future_changed[:2]
unmasked=attend(x[0],x,x)
assert unmasked!=full[0]
print('causal output:',full,'; first row without causal mask:',unmasked)

## 3. 增量缓存与完整因果计算对照

In [ ]:
cache_k=[];cache_v=[];incremental=[]
for token in x:
    cache_k.append(token[:]);cache_v.append(token[:])
    incremental.append(attend(token,cache_k,cache_v))
assert all(math.isclose(a,b,abs_tol=1e-12) for row_a,row_b in zip(full,incremental) for a,b in zip(row_a,row_b))
assert len(cache_k)==len(x)
wrong=attend(x[-1],cache_k[:-1],cache_v[:-1])
assert wrong!=incremental[-1]
print('cache matches full; removing the current KV changes output')

## 4. RoPE：同时平移位置的点积不变

In [ ]:
def rotate(vector,pos,theta=.2):
    angle=pos*theta;c=math.cos(angle);s=math.sin(angle)
    return [c*vector[0]-s*vector[1],s*vector[0]+c*vector[1]]
q=[1.0,0.0];k=[0.0,1.0]
original=dot(rotate(q,3),rotate(k,7))
shifted=dot(rotate(q,8),rotate(k,12))
relative=dot(q,rotate(k,4))
assert math.isclose(original,shifted,abs_tol=1e-12)
assert math.isclose(original,relative,abs_tol=1e-12)
assert math.isclose(dot(q,q),dot(rotate(q,8),rotate(q,8)))
assert not math.isclose(original,dot(rotate(q,3),rotate(k,8)))
print('same relative displacement:',original,shifted)

## 5. GQA 的 KV 基线：与总显存分开

In [ ]:
def kv_bytes(layers,batch,tokens,kv_heads,head_dim,bytes_per_element):
    args=(layers,batch,tokens,kv_heads,head_dim,bytes_per_element)
    if any(type(v) is not int or v<=0 for v in args):raise ValueError('positive integer dimensions required')
    return 2*math.prod(args)
gqa=kv_bytes(32,1,4096,8,128,2)
mha=kv_bytes(32,1,4096,32,128,2)
assert gqa==512*2**20 and mha==2048*2**20
assert rejects(lambda:kv_bytes(32,1,4096,0,128,2))
print('GQA/MHA KV MiB:',gqa/2**20,mha/2**20,'; excludes weights and allocator overhead')

## 修改与记录

改变一个输入，先记录预测，再重新执行有关格。不要直接将检查值改成 True。导入只校验格式，不证明真实模型能力。

In [ ]:
import json
from pathlib import Path
checks = {'stable_softmax': True, 'causal_mask': True, 'cache_equivalence': True, 'rope_relative': True, 'kv_heads': True, 'negative_inputs': True}
report = {'experiment': 'lesson-model-foundations', 'scope': 'stdlib fixtures; no model/service',
          'checks': checks, 'my_explanation': ''}
Path('mechanism_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(json.dumps(report, ensure_ascii=False, indent=2))